# 阶段二：模型训练

**实验目标：**
1. 完成类别编码（独热编码）
2. 完成训练集/测试集划分与数据标准化
3. 训练至少3种模型并完成交叉验证
4. 模型横向对比

In [ ]:
# 环境初始化
import sys
sys.path.append('..')

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

from src.data_loader import load_processed_data
from src.model_utils import prepare_model_data, train_baseline_models, cross_validate_models
from src.evaluation import plot_model_comparison

# 设置目录
DATA_DIR = Path('..') / 'data'
FIG_DIR = Path('..') / 'figures' / 'models'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('✓ 环境初始化完成')

## 1. 加载清洗后的数据

In [ ]:
# 加载清洗后的数据
house = load_processed_data(DATA_DIR)
print(f'数据形状: {house.shape}')
display(house.head())

## 2. 准备建模数据

In [ ]:
# 准备建模数据
X_train, X_test, y_train, y_test, scaler, feature_cols = prepare_model_data(house)

## 3. 训练基线模型

In [ ]:
# 训练基线模型
models, results_df = train_baseline_models(X_train, y_train, X_test, y_test)

In [ ]:
# 绘制模型对比图
plot_model_comparison(results_df, FIG_DIR)

## 4. 交叉验证

In [ ]:
# 5折交叉验证
cv_results = cross_validate_models(models, X_train, y_train, cv=5)
display(cv_results)

## 5. 模型评估可视化

In [ ]:
from src.evaluation import plot_prediction_vs_actual, plot_feature_importance

# 为每个模型绘制预测vs真实值图
for name, model in models.items():
    y_pred = model.predict(X_test)
    plot_prediction_vs_actual(y_test, y_pred, name, FIG_DIR)
    plot_feature_importance(model, feature_cols, name, FIG_DIR, top_n=15)

## 阶段二模型训练总结

**完成的工作：**
1. 对朝向和所在区县进行了独热编码
2. 按8:2比例划分训练集和测试集
3. 使用StandardScaler进行标准化（仅在训练集上fit）
4. 训练了3种模型：线性回归、随机森林、梯度提升
5. 完成了5折交叉验证

**模型对比结论：**
- 线性回归作为基线模型，可解释性强但预测能力有限
- 随机森林和梯度提升作为集成方法，通常能获得更好的预测效果
- 需要根据R²、RMSE、MAE等指标综合评估模型性能